# River Tree

An example workflow for gathering NHD Plus data and generating a tree.  This tree can then be iterated over to accumulate data.  Braided systems are merged into a single reach, and the system is simplified relative to NHD Plus's raw format.

This type of workflow is useful for setting up stream network models.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os,sys
import numpy as np
from matplotlib import pyplot as plt
import shapely
import logging

import watershed_workflow
import watershed_workflow.crs
import watershed_workflow.source_list
import watershed_workflow.config
import watershed_workflow.ui
import watershed_workflow.utils

watershed_workflow.ui.setup_logging(1,None)
crs = watershed_workflow.crs.default_crs


In [ ]:
# open a shapefile for use here
shpfile = os.path.join('Coweeta', 'input_data', 'coweeta_basin.shp')
shp = watershed_workflow.getShapes(shpfile, crs=crs)
shp

In [ ]:
# find the rivers in this shape
reaches = watershed_workflow.getShapesByGeometry(watershed_workflow.source_list.hydrography_sources['NHDv2.1'],
                                                 shp.geometry.iloc[0], shp.crs, crs)
reaches
#for c in reaches.columns:
#    print(c)

In [ ]:
# make the global tree
rivers = watershed_workflow.river_tree.createRiverTrees(reaches, method='hydroseq')

# check that only one tree was formed
assert(len(rivers) == 1)
river = rivers[0]

In [ ]:
ax = shp.boundary.plot(color='k')
reaches.plot(color='r', ax=ax)


In [ ]:
print('NAME: length')
print('-------------')
gnis_name = 'GNIS_Name' if 'GNIS_Name' in river else 'gnis_name'

for r in river:
    print(f"{r.properties[gnis_name]}: {r.properties['shape_length']}")